# Assignment 2 - Reddit AI communities

Same steps as `run_pipeline.py`, split into cells so we can inspect results along the way.


In [ ]:
# Sanity checking — VS Code should have the project root on the Python path when the folder is open
from pathlib import Path

print("Working directory:", Path.cwd())
print("src/ here?", Path("src").is_dir(), "| config.yaml here?", Path("config.yaml").is_file())

You are in: C:\Users\Vihang Mehere\OneDrive\Desktop\Social_Media_A2
src folder here? True  | config.yaml here? True


In [ ]:
import sys

print("Kernel:", sys.executable)

try:
    import pandas 
    print("Dependencies OK.")
except ModuleNotFoundError:
    raise RuntimeError(
        "Packages not installed for this kernel.\n"
        f"Python: {sys.executable}\n"
        "In a terminal: pip install -r requirements.txt"
    )

Kernel: c:\Users\Vihang Mehere\OneDrive\Desktop\Social_Media_A2\venv\Scripts\python.exe


OK — pandas and other libs are available in this kernel.


In [ ]:
import sys
from pathlib import Path


def find_project_root() -> Path:
    start = Path.cwd().resolve()
    if start.name == "notebooks":
        start = start.parent
    for folder in [start, *start.parents]:
        if (folder / "config.yaml").is_file() and (folder / "src").is_dir():
            return folder
    raise FileNotFoundError(
        "Can't find the project folder (need config.yaml and src/ together).\n"
        "In VS Code use File > Open Folder on the assignment project root."
    )


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Using project folder:", ROOT)

from src.config_loader import load_config
from src.preprocess import preprocess
from src.nlp_analysis import run_nlp
from src.network_analysis import run_network
from src.create_sample import create_sample

cfg = load_config(ROOT / "config.yaml")
cfg  # should show subreddits, releases, and output paths

Using project folder: C:\Users\Vihang Mehere\OneDrive\Desktop\Social_Media_A2


{'project': {'title': 'How do AI Communities react to new AI releases',
  'description': 'Compare sentiment, topics, and user interaction networks across AI-focused subreddits around major model release windows.\n'},
 'subreddits': ['MachineLearning',
  'LocalLLaMA',
  'ChatGPT',
  'artificial',
  'OpenAI'],
 'releases': [{'id': 'llama3',
   'name': 'Meta Llama 3',
   'announcement_utc': 1712160000,
   'search_queries': ['Llama 3', 'llama3'],
   'pre_days': 7,
   'post_days': 14},
  {'id': 'gpt4o',
   'name': 'OpenAI GPT-4o',
   'announcement_utc': 1715817600,
   'search_queries': ['GPT-4o', 'gpt4o'],
   'pre_days': 7,
   'post_days': 14},
  {'id': 'claude35',
   'name': 'Anthropic Claude 3.5 Sonnet',
   'announcement_utc': 1718668800,
   'search_queries': ['Claude 3.5', 'claude 3.5 sonnet'],
   'pre_days': 7,
   'post_days': 14}],
 'collection': {'pullpush_base': 'https://api.pullpush.io/reddit/search',
  'submissions_per_query': 150,
  'comments_per_submission': 80,
  'max_submission

In [ ]:
# Only to turn this on if you need to scrape Reddit again (takes a while)
COLLECT = False
SKIP_COMMENTS = False  # True = faster run but networks won't be very meaningful

if COLLECT:
    from src.collect_pullpush import run_collection
    run_collection(cfg, skip_comments=SKIP_COMMENTS)
else:
    print("Using existing data in data/raw/")

Skipping collection — using whatever is already in data/raw/


In [5]:
# Main analysis — same order as run_pipeline.py
preprocess(cfg)
run_nlp(cfg)          # sentiment + topics
run_network(cfg)      # co-comment graphs
create_sample(cfg)    # small CSVs for submission

print("Finished — see outputs/figures and outputs/tables")

Done — check outputs/figures and outputs/tables


In [6]:
import pandas as pd
from IPython.display import display

# Quick look at the tables that went into the report
display(pd.read_csv(ROOT / "outputs/tables/sentiment_by_release_phase_subreddit.csv").head(20))
display(pd.read_csv(ROOT / "outputs/tables/network_summary.csv"))

,release_id,phase,subreddit,n_docs,mean_compound,pct_positive,pct_negative
0,claude35,peak,ChatGPT,25,0.452104,0.640000,0.040000
1,claude35,peak,LocalLLaMA,22,0.436991,0.681818,0.045455
2,claude35,peak,OpenAI,4,0.474775,0.750000,0.000000
3,claude35,peak,artificial,1,0.000000,0.000000,0.000000
4,claude35,post,ChatGPT,375,0.276321,0.584000,0.205333
5,claude35,post,LocalLLaMA,87,0.217867,0.563218,0.287356
6,claude35,post,MachineLearning,3,0.162933,0.666667,0.333333
7,claude35,post,OpenAI,82,0.361820,0.670732,0.121951
8,claude35,post,artificial,5,0.315120,0.600000,0.000000
9,claude35,pre,ChatGPT,1,0.738300,1.000000,0.000000


,group_type,group,nodes,edges,density,avg_clustering,modularity
0,phase,peak,155,2083,0.174529,0.326044,0.315532
1,phase,post,476,5943,0.052570,0.245220,0.800054
2,phase,pre,68,1511,0.663301,0.970588,0.033975
3,release,claude35,277,4328,0.113221,0.326758,0.737333
4,release,gpt4o,394,5046,0.065176,0.246395,0.726316
5,release,llama3,24,157,0.568841,1.000000,0.231734
